# Conexión a base de datos

In [15]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

# Tablas

In [17]:
query = """SELECT * FROM books LIMIT 10"""

pd.io.sql.read_sql(query, con = engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268
5,6,257,1st to Die (Women's Murder Club #1),424,2005-05-20,116
6,7,258,2nd Chance (Women's Murder Club #2),400,2005-05-20,116
7,8,260,4th of July (Women's Murder Club #4),448,2006-06-01,318
8,9,563,A Beautiful Mind,461,2002-02-04,104
9,10,445,A Bend in the Road,341,2005-04-01,116


In [18]:
query = """SELECT * FROM authors LIMIT 10"""

pd.io.sql.read_sql(query, con = engine)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd
5,6,Alan Paton
6,7,Albert Camus/Justin O'Brien
7,8,Aldous Huxley
8,9,Aldous Huxley/Christopher Hitchens
9,10,Aleksandr Solzhenitsyn/H.T. Willetts


In [19]:
query = """SELECT * FROM publishers LIMIT 10"""

pd.io.sql.read_sql(query, con = engine)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company
5,6,Aladdin
6,7,Aladdin Paperbacks
7,8,Albin Michel
8,9,Alfred A. Knopf
9,10,Alfred A. Knopf Books for Young Readers


In [20]:
query = """SELECT * FROM ratings LIMIT 10"""

pd.io.sql.read_sql(query, con = engine)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2
5,6,3,johnsonamanda,4
6,7,3,scotttamara,5
7,8,3,lesliegibbs,5
8,9,4,abbottjames,5
9,10,4,valenciaanne,4


In [21]:
query = """SELECT * FROM reviews LIMIT 10"""

pd.io.sql.read_sql(query, con = engine)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...
5,6,3,lesliegibbs,Analysis no several cause international.
6,7,4,valenciaanne,One there cost another. Say type save. With pe...
7,8,4,abbottjames,Within enough mother. There at system full rec...
8,9,5,npowers,Thank now focus realize economy focus fly. Ite...
9,10,5,staylor,Game push lot reduce where remember. Including...


# Ejercicios
Libros publicados después del 1 de enero de 2000

In [22]:
query = """SELECT COUNT(*) FROM books
           WHERE publication_date > '2000-01-01'"""
pd.io.sql.read_sql(query, con = engine)

,count
0,819


En total hay 819 libros que se publicaron después del 1 de enero del 2000

#  Número de reseñas de usuarios y la calificación promedio para cada libro.

In [24]:
query = """SELECT title, AVG(rating)

           FROM ratings

           INNER JOIN books

           ON ratings.book_id = books.book_id

           GROUP BY title"""

pd.io.sql.read_sql(query, con = engine)

,title,avg
0,The Count of Monte Cristo,4.217391
1,Count Zero (Sprawl #2),2.500000
2,The Botany of Desire: A Plant's-Eye View of th...,3.500000
3,The Poisonwood Bible,4.363636
4,The Canterbury Tales,3.333333
...,...,...
994,Of Love and Other Demons,4.500000
995,In the Heart of the Sea: The Tragedy of the Wh...,3.333333
996,Welcome to Temptation (Dempseys #1),5.000000
997,World's End (The Sandman #8),4.500000


# Editorial que ha publicado el mayor número de libros con más de 50 páginas

In [25]:
query = """SELECT publisher, COUNT(book_id) AS num_books

           FROM publishers

           INNER JOIN books

           ON publishers.publisher_id = books.publisher_id

           WHERE num_pages > 50

           GROUP BY publisher

           ORDER BY num_books DESC

           LIMIT 1"""

pd.io.sql.read_sql(query, con = engine)

,publisher,num_books
0,Penguin Books,42


La editorial que ha publicado el mayor número de libros con más de 50 páginas es Penguin Book

 # Autor que tiene la más alta calificación promedio

In [28]:
query = """
WITH books_50 AS (
    SELECT 
        book_id,
        AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating) >= 50
),

author_avg AS (
    SELECT 
        books.author_id,
        AVG(books_50.avg_rating) AS author_avg_rating
    FROM books
    INNER JOIN books_50
        ON books.book_id = books_50.book_id
    GROUP BY books.author_id
)

SELECT 
    authors.author,
    author_avg.author_avg_rating
FROM author_avg
INNER JOIN authors
    ON authors.author_id = author_avg.author_id
ORDER BY author_avg.author_avg_rating DESC
LIMIT 1"""

pd.io.sql.read_sql(query, con=engine)

,author,author_avg_rating
0,J.K. Rowling/Mary GrandPré,4.283844


# Número promedio de reseñas de texto entre los usuarios

In [29]:
query = """
WITH active_users AS (
    SELECT 
        username
    FROM ratings
    GROUP BY username
    HAVING COUNT(book_id) > 50
),

reviews_count AS (
    SELECT 
        reviews.username,
        COUNT(reviews.text) AS num_reviews
    FROM reviews
    INNER JOIN active_users
        ON reviews.username = active_users.username
    GROUP BY reviews.username
)

SELECT 
    AVG(num_reviews) AS avg_reviews
FROM reviews_count"""

pd.io.sql.read_sql(query, con=engine)

,avg_reviews
0,24.333333


 El número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros es de 24.3 reseñas.